# QLoRA: дообучение Qwen3.5-2B-Base в 4-битном NF4

В этом блокноте QLoRA применяется к `Qwen/Qwen3.5-2B-Base` для бинарной классификации тональности SST-2.

Базовая модель загружается в 4-битном формате **NF4** через bitsandbytes, её исходные веса остаются замороженными, а обучение выполняется только для LoRA-адаптера. До и после обучения модель оценивается одинаковыми способами, чтобы отдельно измерить качество классификации и соблюдение требуемого формата ответа.

## Теоретическая часть

### Что такое QLoRA

**QLoRA** сочетает 4-битную квантизацию базовой модели и LoRA. Квантизованные веса используются в прямом и обратном проходах, но сами не обновляются. Обучаются только небольшие низкоранговые матрицы LoRA.

В упрощённом виде вычисление можно представить как:

$y = W_{4bit}x + \frac{\alpha}{r}BAx$

где $W_{4bit}$ — замороженные квантизованные веса, а $A$ и $B$ — обучаемые матрицы LoRA.

### Чем QLoRA отличается от обычной LoRA

| Свойство | LoRA | QLoRA |
|---|---|---|
| Базовые веса | BF16 / FP16 / FP32 | 4-битные |
| Обучаемые параметры | LoRA-адаптер | LoRA-адаптер |
| Базовые веса обновляются | нет | нет |
| Основная цель | параметрически эффективное дообучение | параметрически и по памяти эффективное дообучение |
| Типичная 4-битная схема | не требуется | NF4 |

### Почему используется NF4

`NF4` — 4-битный формат NormalFloat, введённый в работе QLoRA. Он рассчитан на распределения весов, близкие к нормальному, и предназначен именно для 4-битного обучения базовых моделей.

В Transformers схема задаётся через `BitsAndBytesConfig` параметром `bnb_4bit_quant_type="nf4"`.

### Двойная квантизация

Параметр `bnb_4bit_use_double_quant=True` включает дополнительное сжатие коэффициентов масштаба, которые сами используются для восстановления 4-битных весов. Это уменьшает служебные затраты представления без изменения логики LoRA.

### BF16 как формат вычислений

Хранение весов в NF4 не означает, что все матричные операции выполняются в 4 битах. В этом блокноте `bnb_4bit_compute_dtype=torch.bfloat16`, поэтому вычисления выполняются в BF16, а 4-битный формат используется для компактного хранения базовых весов.

### Подготовка модели к k-bit обучению

`prepare_model_for_kbit_training()` подготавливает квантизованную Transformers-модель к обучению через PEFT. В частности, базовые параметры замораживаются, а чувствительные к точности части модели приводятся к подходящему формату вычислений.

После этой подготовки к модели добавляется LoRA-адаптер.

### Почему `target_modules="all-linear"`

Для QLoRA официальный подход PEFT рекомендует применять LoRA ко всем линейным слоям трансформера. Значение `target_modules="all-linear"` позволяет не перечислять архитектурно-зависимые имена проекций вручную.

Для `PreTrainedModel` выходной слой при таком выборе исключается PEFT автоматически.

### Что именно обучается

После создания PEFT-модели параметры базовой 4-битной модели должны оставаться замороженными. Градиенты получают только параметры LoRA.

Для слоя с входной размерностью $d_{in}$ и выходной размерностью $d_{out}$ LoRA добавляет примерно

$r(d_{in}+d_{out})$

обучаемых параметров вместо полного обновления матрицы размером $d_{in}d_{out}$.

### Paged AdamW

В обучении используется `paged_adamw_32bit` из bitsandbytes. Paged-оптимизаторы уменьшают риск кратковременных пиков потребления памяти за счёт постраничного управления состояниями оптимизатора.

Для небольшого 2B-эксперимента это не является обязательным условием QLoRA, но сохраняет характерную для этого подхода конфигурацию обучения.

### Что будем измерять

До и после QLoRA используются одинаковые метрики:

- Generation Accuracy;
- Forced-Choice Accuracy;
- Generation Macro F1;
- Forced-Choice Macro F1;
- Valid Output Rate;
- Precision, Recall и F1 по классам;
- матрица ошибок;
- Validation Loss, Label Token Accuracy и Perplexity во время обучения.

Generation-based и Forced-Choice оценки разделяют способность модели понять задачу и способность выдать ответ строго в требуемом формате.

### Теоретические источники

- QLoRA: Efficient Finetuning of Quantized LLMs — https://arxiv.org/abs/2305.14314
- PEFT 0.20.0: Quantization — https://huggingface.co/docs/peft/v0.20.0/developer_guides/quantization
- PEFT: LoRA API — https://huggingface.co/docs/peft/main/package_reference/lora
- Transformers: bitsandbytes — https://huggingface.co/docs/transformers/main/quantization/bitsandbytes
- bitsandbytes: 4-bit quantization — https://huggingface.co/docs/bitsandbytes/main/reference/nn/linear4bit

## Практическая часть

### 1. Импорты и проверка среды

Зависимости устанавливаются в Docker-окружении через `requirements.txt`. Внутри блокнота пакеты не устанавливаются и минимальные версии через `assert` не проверяются.

In [1]:
import gc
import json
import sys
from contextlib import contextmanager
from dataclasses import dataclass
from importlib.metadata import version as package_version
from pathlib import Path
from typing import Any

import bitsandbytes as bnb
import numpy as np
import plotly.graph_objects as go
import torch
from datasets import DatasetDict, load_dataset
from huggingface_hub import HfApi, create_repo
from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassConfusionMatrix,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

print(f"Python:           {sys.version.split()[0]}")
print(f"PyTorch:          {torch.__version__}")
print(f"Transformers:     {package_version('transformers')}")
print(f"PEFT:             {package_version('peft')}")
print(f"bitsandbytes:     {package_version('bitsandbytes')}")
print(f"Datasets:         {package_version('datasets')}")
print(f"Accelerate:       {package_version('accelerate')}")
print(f"TorchMetrics:     {package_version('torchmetrics')}")
print(f"Hugging Face Hub: {package_version('huggingface_hub')}")
print(f"Plotly:           {package_version('plotly')}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "Этот блокнот рассчитан на обучение QLoRA с CUDA."
    )

device = torch.device(
    f"cuda:{torch.cuda.current_device()}"
)

Python:           3.10.12
PyTorch:          2.13.0+cu130
Transformers:     5.14.1
PEFT:             0.20.0
bitsandbytes:     0.50.1
Datasets:         5.0.1
Accelerate:       1.14.0
TorchMetrics:     1.9.0
Hugging Face Hub: 1.28.0
Plotly:           6.9.0


### 2. Конфигурация

`RUN_MODE="smoke"` использует 8 000 обучающих примеров и предназначен для полного проверочного запуска. `RUN_MODE="full"` использует весь обучающий раздел SST-2.

Валидационный раздел не сокращается, поэтому исходная 4-битная модель и QLoRA сравниваются на одной и той же полной выборке.

In [2]:
SEED = 42

MODEL_ID = "Qwen/Qwen3.5-2B-Base"
DATASET_ID = "stanfordnlp/sst2"

MODEL_REVISION = None
DATASET_REVISION = None

TEXT_COLUMN = "sentence"
LABEL_COLUMN = "label"
LABEL_NAMES = {
    0: "negative",
    1: "positive",
}

RUN_MODE = "smoke"
assert RUN_MODE in {"smoke", "full"}

MAX_LENGTH = 128
MAX_TRAIN_SAMPLES = 8_000 if RUN_MODE == "smoke" else None
MAX_EVAL_SAMPLES = None

BASELINE_EVAL_SAMPLES = None
FINAL_EVAL_SAMPLES = None
GENERATION_BATCH_SIZE = 32
FORCED_CHOICE_BATCH_SIZE = 16
MAX_NEW_TOKENS = 4

ARTIFACT_RELOAD_SAMPLES = 8
RUN_ARTIFACT_RELOAD_TEST = True

BNB_QUANT_TYPE = "nf4"
BNB_COMPUTE_DTYPE = torch.bfloat16
BNB_USE_DOUBLE_QUANT = True

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "all-linear"

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 2e-4
LR_SCHEDULER_TYPE = "linear"
WARMUP_RATIO = 0.05
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.0
EVAL_STEPS = 100
SAVE_STEPS = EVAL_STEPS
WEIGHT_DECAY = 0.0
USE_GRADIENT_CHECKPOINTING = False

PUSH_TO_HUB = False
HF_NAMESPACE = "artyomboyko"
HUB_MODEL_ID = f"{HF_NAMESPACE}/qwen3.5-2b-sst2-qlora"

cwd = Path.cwd().resolve()

if cwd.name == "peft":
    NOTEBOOK_DIR = cwd
elif (cwd / "notebooks" / "finetuning" / "peft").is_dir():
    NOTEBOOK_DIR = (cwd / "notebooks" / "finetuning" / "peft").resolve()
elif Path("/workspace/notebooks/finetuning/peft").is_dir():
    NOTEBOOK_DIR = Path("/workspace/notebooks/finetuning/peft")
else:
    NOTEBOOK_DIR = cwd

OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "qwen3.5-2b-sst2-qlora"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
torch.set_float32_matmul_precision("high")

RESULTS_TABLE = []

print(f"Notebook directory:   {NOTEBOOK_DIR}")
print(f"Output directory:     {OUTPUT_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

Notebook directory:   /workspace/notebooks/finetuning/peft
Output directory:     /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-qlora
Checkpoint directory: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-qlora/checkpoints


### 3. Загрузка SST-2

Используется Stanford SST-2 с двумя классами: `negative` и `positive`. Обучающая выборка может сокращаться в режиме `smoke`, а валидационная выборка сохраняется полностью.

In [3]:
raw_dataset = load_dataset(
    DATASET_ID,
    revision=DATASET_REVISION,
)

dataset = DatasetDict(
    train=raw_dataset["train"],
    validation=raw_dataset["validation"],
)

def limit_split(split, max_samples):
    if max_samples is None:
        return split

    count = min(max_samples, len(split))
    return split.shuffle(seed=SEED).select(range(count))

dataset["train"] = limit_split(
    dataset["train"],
    MAX_TRAIN_SAMPLES,
)

dataset["validation"] = limit_split(
    dataset["validation"],
    MAX_EVAL_SAMPLES,
)

print(dataset)
print(dataset["train"][0])

print(f"\nTrain samples:      {len(dataset['train']):,}")
print(
    f"Validation samples: {len(dataset['validation']):,} "
    f"/ {len(raw_dataset['validation']):,}"
)

if MAX_EVAL_SAMPLES is None:
    assert len(dataset["validation"]) == len(raw_dataset["validation"])

dataset_train_fingerprint = dataset["train"]._fingerprint
dataset_validation_fingerprint = dataset["validation"]._fingerprint

try:
    resolved_dataset_revision = (
        DATASET_REVISION
        or HfApi().dataset_info(DATASET_ID).sha
    )
except Exception:
    resolved_dataset_revision = DATASET_REVISION or "unavailable"

print(f"Dataset revision:   {resolved_dataset_revision}")
print(f"Train fingerprint:  {dataset_train_fingerprint}")
print(f"Valid fingerprint:  {dataset_validation_fingerprint}")

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 8000
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
})
{'idx': 32326, 'sentence': 'klein , charming in comedies like american pie and dead-on in election , ', 'label': 1}

Train samples:      8,000
Validation samples: 872 / 872
Dataset revision:   8d51e7e4887a4caaa95b3fbebbf53c0490b58bbb
Train fingerprint:  637fb4ba0a441ab8
Valid fingerprint:  c1ddc6497ec97f98


### 4. Токенизатор и 4-битная конфигурация NF4

Базовая модель загружается сразу в 4-битном NF4. Включена двойная квантизация, а вычисления выполняются в BF16.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=BNB_QUANT_TYPE,
    bnb_4bit_compute_dtype=BNB_COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=BNB_USE_DOUBLE_QUANT,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.bfloat16,
    quantization_config=quantization_config,
    device_map={"": torch.cuda.current_device()},
)

assert getattr(base_model, "is_loaded_in_4bit", False)

resolved_model_revision = (
    MODEL_REVISION
    or getattr(base_model.config, "_commit_hash", None)
)

if not resolved_model_revision:
    try:
        resolved_model_revision = HfApi().model_info(
            MODEL_ID
        ).sha
    except Exception:
        resolved_model_revision = "unavailable"

print(f"Loaded class:      {type(base_model).__name__}")
print(f"Quantization:      {BNB_QUANT_TYPE.upper()}")
print(f"Compute dtype:     {BNB_COMPUTE_DTYPE}")
print(f"Double quant:      {BNB_USE_DOUBLE_QUANT}")
print(f"Vocabulary:        {len(tokenizer):,}")
print(f"Model revision:    {resolved_model_revision}")

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded class:      Qwen3_5ForCausalLM
Quantization:      NF4
Compute dtype:     torch.bfloat16
Double quant:      True
Vocabulary:        248,077
Model revision:    b1485b2fa6dfa1287294f269f5fb618e03d52d7c


### 5. Проверка 4-битных слоёв

После загрузки обычные линейные слои, поддерживаемые bitsandbytes, должны быть представлены как `Linear4bit`. Эта проверка подтверждает, что базовая модель действительно загружена в 4-битном режиме.

In [5]:
linear4bit_modules = [
    name
    for name, module in base_model.named_modules()
    if isinstance(module, bnb.nn.Linear4bit)
]

assert linear4bit_modules

print(f"Linear4bit modules: {len(linear4bit_modules)}")
print("\nFirst Linear4bit modules:")

for name in linear4bit_modules[:20]:
    print(name)

Linear4bit modules: 186

First Linear4bit modules:
model.layers.0.linear_attn.out_proj
model.layers.0.linear_attn.in_proj_qkv
model.layers.0.linear_attn.in_proj_z
model.layers.0.linear_attn.in_proj_b
model.layers.0.linear_attn.in_proj_a
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.1.linear_attn.out_proj
model.layers.1.linear_attn.in_proj_qkv
model.layers.1.linear_attn.in_proj_z
model.layers.1.linear_attn.in_proj_b
model.layers.1.linear_attn.in_proj_a
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.2.linear_attn.out_proj
model.layers.2.linear_attn.in_proj_qkv
model.layers.2.linear_attn.in_proj_z
model.layers.2.linear_attn.in_proj_b


### 6. Формат задачи

SST-2 формулируется как задача causal language modeling. Loss рассчитывается только по токенам целевой метки `negative` или `positive`; токены инструкции и текста отзыва маскируются значением `-100`.

In [6]:
VISIBLE_INSTRUCTION = (
    "Classify the sentiment of this movie review as positive or negative."
)

def build_prompt(text):
    return (
        f"{VISIBLE_INSTRUCTION}\n"
        f"Review: {text.strip()}\n"
        "Sentiment:"
    )

def preprocess_batch(examples):
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for text, label_id in zip(
        examples[TEXT_COLUMN],
        examples[LABEL_COLUMN],
    ):
        target_text = " " + LABEL_NAMES[int(label_id)]

        target_ids = tokenizer(
            target_text,
            add_special_tokens=False,
        )["input_ids"]

        target_ids = target_ids + [tokenizer.eos_token_id]

        max_prompt_length = max(
            1,
            MAX_LENGTH - len(target_ids),
        )

        prompt_ids = tokenizer(
            build_prompt(text),
            add_special_tokens=False,
            truncation=True,
            max_length=max_prompt_length,
        )["input_ids"]

        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids

        all_input_ids.append(input_ids)
        all_attention_masks.append([1] * len(input_ids))
        all_labels.append(labels)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels,
    }

processed_dataset = dataset.map(
    preprocess_batch,
    batched=True,
    batch_size=1_000,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing SST-2",
)

print(processed_dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 872
    })
})


### 7. Динамический padding

Последовательности дополняются до максимальной длины текущего батча. Позиции padding в `labels` получают `-100` и не участвуют в расчёте loss.

In [7]:
@dataclass
class CausalClassificationCollator:
    tokenizer: Any

    def __call__(self, features):
        model_features = [
            {
                "input_ids": feature["input_ids"],
                "attention_mask": feature["attention_mask"],
            }
            for feature in features
        ]

        batch = self.tokenizer.pad(
            model_features,
            padding=True,
            return_tensors="pt",
        )

        sequence_length = batch["input_ids"].shape[1]

        padded_labels = []

        for feature in features:
            labels = feature["labels"]
            padding_length = sequence_length - len(labels)
            padded_labels.append(
                labels + [-100] * padding_length
            )

        batch["labels"] = torch.tensor(
            padded_labels,
            dtype=torch.long,
        )

        return batch

data_collator = CausalClassificationCollator(
    tokenizer=tokenizer,
)

test_batch = data_collator(
    [
        processed_dataset["train"][0],
        processed_dataset["train"][1],
    ]
)

for key, value in test_batch.items():
    print(f"{key:16s}: {tuple(value.shape)} {value.dtype}")

input_ids       : (2, 37) torch.int64
attention_mask  : (2, 37) torch.int64
labels          : (2, 37) torch.int64


### 8. Метрики и функции оценки

Для итогового сравнения используются TorchMetrics и два независимых режима оценки. При свободной генерации модель должна самостоятельно вывести допустимую метку. При принудительном выборе сравниваются вероятности только двух допустимых меток.

In [8]:
CLASS_LABELS = ("negative", "positive")
INVALID_LABEL_ID = len(CLASS_LABELS)
NUM_EVAL_CLASSES = INVALID_LABEL_ID + 1

EVAL_METRICS = MetricCollection(
    {
        "accuracy": MulticlassAccuracy(NUM_EVAL_CLASSES, average="micro"),
        "precision": MulticlassPrecision(
            NUM_EVAL_CLASSES,
            average=None,
            zero_division=0,
        ),
        "recall": MulticlassRecall(
            NUM_EVAL_CLASSES,
            average=None,
            zero_division=0,
        ),
        "f1": MulticlassF1Score(
            NUM_EVAL_CLASSES,
            average=None,
            zero_division=0,
        ),
        "confusion_matrix": MulticlassConfusionMatrix(
            NUM_EVAL_CLASSES,
        ),
    }
)

_label_token_ids = [
    tokenizer(
        f" {label}",
        add_special_tokens=False,
    )["input_ids"]
    for label in CLASS_LABELS
]

if not all(len(ids) == 1 for ids in _label_token_ids):
    raise ValueError(
        "Forced-choice evaluation requires single-token labels."
    )

LABEL_TOKEN_IDS = torch.tensor(
    [ids[0] for ids in _label_token_ids],
    dtype=torch.long,
)


@contextmanager
def temporary_padding_side(tokenizer, padding_side):
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = padding_side
    try:
        yield
    finally:
        tokenizer.padding_side = original_padding_side


def select_eval_split(raw_split, max_samples):
    count = (
        len(raw_split)
        if max_samples is None
        else min(max_samples, len(raw_split))
    )
    return raw_split.select(range(count))


def tokenize_eval_batch(
    texts,
    device,
    *,
    max_length=MAX_LENGTH,
    add_special_tokens=True,
):
    return tokenizer(
        [build_prompt(text) for text in texts],
        add_special_tokens=add_special_tokens,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    ).to(device)


def normalize_prediction(text):
    text = text.strip().lower()
    return next(
        (
            index
            for index, label in enumerate(CLASS_LABELS)
            if text.startswith(label)
        ),
        INVALID_LABEL_ID,
    )


def classification_metrics(predictions, references):
    predictions = torch.as_tensor(
        predictions,
        dtype=torch.long,
    )
    references = torch.as_tensor(
        references,
        dtype=torch.long,
    )

    values = EVAL_METRICS.clone()(
        predictions,
        references,
    )

    precision = values["precision"][: len(CLASS_LABELS)]
    recall = values["recall"][: len(CLASS_LABELS)]
    f1 = values["f1"][: len(CLASS_LABELS)]

    return {
        "accuracy": values["accuracy"].item(),
        "macro_f1": f1.mean().item(),
        "per_class": {
            label: {
                "precision": precision[index].item(),
                "recall": recall[index].item(),
                "f1": f1[index].item(),
            }
            for index, label in enumerate(CLASS_LABELS)
        },
        "confusion_matrix": (
            values["confusion_matrix"][: len(CLASS_LABELS)]
            .to(torch.int64)
            .tolist()
        ),
        "total": references.numel(),
    }


def print_classification_summary(title, metrics):
    print(f"\n{title}\n{'-' * len(title)}")
    print(f"Accuracy:  {metrics['accuracy']:.2%}")
    print(f"Macro F1:  {metrics['macro_f1']:.4f}")

    for label in CLASS_LABELS:
        values = metrics["per_class"][label]
        print(
            f"{label:8s} "
            f"precision={values['precision']:.4f} "
            f"recall={values['recall']:.4f} "
            f"f1={values['f1']:.4f}"
        )

    if "valid_output_rate" in metrics:
        print(
            "Valid output rate: "
            f"{metrics['valid_output_rate']:.2%}"
        )

    print(
        "\nConfusion matrix "
        "(rows=actual, columns=predicted)"
    )
    print(
        f"{'':12s}"
        f"{'negative':>10s}"
        f"{'positive':>10s}"
        f"{'invalid':>10s}"
    )

    for label, row in zip(
        CLASS_LABELS,
        metrics["confusion_matrix"],
    ):
        print(
            f"{label:12s}"
            f"{row[0]:10d}"
            f"{row[1]:10d}"
            f"{row[2]:10d}"
        )


def evaluate_generation(
    model,
    raw_split,
    max_samples=None,
    batch_size=32,
):
    model.eval()
    eval_split = select_eval_split(
        raw_split,
        max_samples,
    )
    model_device = next(model.parameters()).device
    predictions, references, examples = [], [], []

    with temporary_padding_side(
        tokenizer,
        "left",
    ), torch.inference_mode():
        for start in range(
            0,
            len(eval_split),
            batch_size,
        ):
            batch = eval_split[
                start:start + batch_size
            ]

            inputs = tokenize_eval_batch(
                batch[TEXT_COLUMN],
                model_device,
            )

            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            generated_texts = tokenizer.batch_decode(
                outputs[
                    :,
                    inputs["input_ids"].shape[1]:,
                ],
                skip_special_tokens=True,
            )

            batch_predictions = [
                normalize_prediction(text)
                for text in generated_texts
            ]
            batch_references = [
                int(label_id)
                for label_id in batch[LABEL_COLUMN]
            ]

            predictions.extend(batch_predictions)
            references.extend(batch_references)

            examples.extend(
                (
                    text,
                    (
                        None
                        if prediction == INVALID_LABEL_ID
                        else CLASS_LABELS[prediction]
                    ),
                    CLASS_LABELS[reference],
                )
                for text, prediction, reference in zip(
                    generated_texts,
                    batch_predictions,
                    batch_references,
                )
            )

    metrics = classification_metrics(
        predictions,
        references,
    )
    metrics.update(
        {
            "valid_output_rate": (
                sum(
                    prediction != INVALID_LABEL_ID
                    for prediction in predictions
                )
                / len(eval_split)
            ),
            "predictions": predictions,
            "references": references,
            "examples": examples,
        }
    )
    return metrics


def evaluate_forced_choice(
    model,
    raw_split,
    max_samples=None,
    batch_size=16,
):
    model.eval()
    eval_split = select_eval_split(
        raw_split,
        max_samples,
    )
    model_device = next(model.parameters()).device
    label_token_ids = LABEL_TOKEN_IDS.to(
        model_device
    )
    predictions, references = [], []

    with temporary_padding_side(
        tokenizer,
        "left",
    ), torch.inference_mode():
        for start in range(
            0,
            len(eval_split),
            batch_size,
        ):
            batch = eval_split[
                start:start + batch_size
            ]

            inputs = tokenize_eval_batch(
                batch[TEXT_COLUMN],
                model_device,
                max_length=MAX_LENGTH - 1,
                add_special_tokens=False,
            )

            label_logits = (
                model(**inputs)
                .logits[:, -1, :]
                .index_select(
                    -1,
                    label_token_ids,
                )
            )

            predictions.extend(
                label_logits.argmax(dim=-1).tolist()
            )
            references.extend(
                int(label_id)
                for label_id in batch[LABEL_COLUMN]
            )

    metrics = classification_metrics(
        predictions,
        references,
    )
    metrics.update(
        {
            "predictions": predictions,
            "references": references,
        }
    )
    return metrics

### 9. Оценка исходной 4-битной модели

До добавления LoRA оценивается уже квантизованная NF4-модель. Именно она является корректной исходной точкой для QLoRA: после обучения меняется только адаптер, а 4-битные базовые веса остаются теми же.

In [9]:
baseline_generation_metrics = evaluate_generation(
    base_model,
    dataset["validation"],
    max_samples=BASELINE_EVAL_SAMPLES,
    batch_size=GENERATION_BATCH_SIZE,
)

baseline_forced_choice_metrics = evaluate_forced_choice(
    base_model,
    dataset["validation"],
    max_samples=BASELINE_EVAL_SAMPLES,
    batch_size=FORCED_CHOICE_BATCH_SIZE,
)

assert baseline_generation_metrics["total"] == len(dataset["validation"])
assert baseline_forced_choice_metrics["total"] == len(dataset["validation"])

print_classification_summary(
    "NF4 base — generation-based evaluation",
    baseline_generation_metrics,
)

print_classification_summary(
    "NF4 base — forced-choice evaluation",
    baseline_forced_choice_metrics,
)

print("\nGeneration examples:")

for generated, prediction, reference in baseline_generation_metrics["examples"][:10]:
    print(
        f"generated={generated!r:20s} "
        f"parsed={prediction!r:10s} "
        f"reference={reference}"
    )

RESULTS_TABLE.append(
    {
        "variant": "Исходная NF4",
        "generation_accuracy": baseline_generation_metrics["accuracy"],
        "forced_choice_accuracy": baseline_forced_choice_metrics["accuracy"],
        "generation_macro_f1": baseline_generation_metrics["macro_f1"],
        "forced_choice_macro_f1": baseline_forced_choice_metrics["macro_f1"],
        "valid_output_rate": baseline_generation_metrics["valid_output_rate"],
    }
)


NF4 base — generation-based evaluation
--------------------------------------
Accuracy:  0.00%
Macro F1:  0.0000
negative precision=0.0000 recall=0.0000 f1=0.0000
positive precision=0.0000 recall=0.0000 f1=0.0000
Valid output rate: 0.00%

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative             0         0       428
positive             0         0       444

NF4 base — forced-choice evaluation
-----------------------------------
Accuracy:  50.92%
Macro F1:  0.3374
negative precision=0.0000 recall=0.0000 f1=0.0000
positive precision=0.5092 recall=1.0000 f1=0.6748

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative             0       428         0
positive             0       444         0

Generation examples:
generated='\n\n<think>\nHmm'   parsed=None       reference=positive
generated='\n\n<think>\nHmm'   parsed=None       reference=negative
generated='\n\n<think>\nHmm'   par

### 10. Подготовка модели к k-bit обучению

`prepare_model_for_kbit_training()` подготавливает квантизованную модель для обучения PEFT. После этой операции базовые параметры должны оставаться замороженными.

In [10]:
base_model.config.use_cache = False

model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
)

base_trainable_before_lora = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

assert base_trainable_before_lora == 0

print(
    "Trainable parameters before LoRA:",
    f"{base_trainable_before_lora:,}",
)

Trainable parameters before LoRA: 0


### 11. Создание конфигурации LoRA

QLoRA использует обычный `LoraConfig`, но применяет его к 4-битной базовой модели. `target_modules="all-linear"` добавляет LoRA ко всем поддерживаемым линейным слоям.

In [11]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

lora_config

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules='all-linear', exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)

### 12. Создание QLoRA-модели

`get_peft_model()` добавляет LoRA-матрицы поверх уже квантизованных линейных слоёв. После этого обучаемыми должны быть только параметры адаптера.

In [12]:
model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_fraction = (
    trainable_parameters / total_parameters
)

assert trainable_parameters > 0
assert trainable_fraction < 0.1

print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Total parameters:     {total_parameters:,}")
print(f"Trainable fraction:   {trainable_fraction:.4%}")

trainable params: 16,819,200 || all params: 1,898,644,288 || trainable%: 0.8859
Trainable parameters: 16,819,200
Total parameters:     1,212,285,760
Trainable fraction:   1.3874%


### 13. Проверка внедрённых LoRA-модулей

Проверяем, что PEFT действительно добавил `lora_A` и `lora_B` в линейные слои модели.

In [13]:
adapted_modules = [
    name
    for name, module in model.named_modules()
    if hasattr(module, "lora_A")
    and hasattr(module, "lora_B")
]

assert adapted_modules

print(f"Adapted modules: {len(adapted_modules)}")
print("\nFirst adapted modules:")

for name in adapted_modules[:20]:
    print(name)

Adapted modules: 186

First adapted modules:
base_model.model.model.layers.0.linear_attn.out_proj
base_model.model.model.layers.0.linear_attn.in_proj_qkv
base_model.model.model.layers.0.linear_attn.in_proj_z
base_model.model.model.layers.0.linear_attn.in_proj_b
base_model.model.model.layers.0.linear_attn.in_proj_a
base_model.model.model.layers.0.mlp.gate_proj
base_model.model.model.layers.0.mlp.up_proj
base_model.model.model.layers.0.mlp.down_proj
base_model.model.model.layers.1.linear_attn.out_proj
base_model.model.model.layers.1.linear_attn.in_proj_qkv
base_model.model.model.layers.1.linear_attn.in_proj_z
base_model.model.model.layers.1.linear_attn.in_proj_b
base_model.model.model.layers.1.linear_attn.in_proj_a
base_model.model.model.layers.1.mlp.gate_proj
base_model.model.model.layers.1.mlp.up_proj
base_model.model.model.layers.1.mlp.down_proj
base_model.model.model.layers.2.linear_attn.out_proj
base_model.model.model.layers.2.linear_attn.in_proj_qkv
base_model.model.model.layers.2.

### 14. Проверка замороженных базовых весов

Все обучаемые параметры должны относиться к LoRA. Если обнаружится другой обучаемый параметр, выполнение останавливается.

In [14]:
unexpected_trainable = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
    and "lora_" not in name
]

assert not unexpected_trainable, unexpected_trainable

print("Only LoRA parameters are trainable.")

Only LoRA parameters are trainable.


### 15. TrainingArguments

Используется линейное уменьшение learning rate, разогрев на первых 5% шагов, промежуточная оценка каждые 100 optimizer steps и Early Stopping по `eval_loss`.

Оптимизатор — `paged_adamw_32bit`. При завершении обучения `Trainer` восстанавливает сохранённый чекпойнт с минимальным validation loss.

In [15]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    logging_strategy="steps",
    logging_steps=20,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,

    bf16=True,
    tf32=True,

    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs=(
        {"use_reentrant": False}
        if USE_GRADIENT_CHECKPOINTING
        else None
    ),

    optim="paged_adamw_32bit",

    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,

    include_for_metrics=["loss"],
    report_to="none",
    push_to_hub=False,
)

print("Run mode:", RUN_MODE)
print("Training examples:", len(dataset["train"]))
print("Effective batch size:", TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Warmup ratio:", WARMUP_RATIO)
print("Optimizer:", training_args.optim)
print("Eval interval:", EVAL_STEPS, "optimizer steps")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Run mode: smoke
Training examples: 8000
Effective batch size: 16
Learning rate: 0.0002
Warmup ratio: 0.05
Optimizer: OptimizerNames.PAGED_ADAMW
Eval interval: 100 optimizer steps


### 16. Метрики промежуточной валидации

Во время `Trainer`-валидации вычисляются Label Token Accuracy и Perplexity. Для экономии объёма данных полные логиты заменяются на `argmax` до передачи в `compute_metrics`.

In [16]:
def preprocess_logits_for_metrics(
    logits,
    labels,
):
    if isinstance(logits, tuple):
        logits = logits[0]

    return logits.argmax(dim=-1)


def compute_trainer_metrics(eval_prediction):
    predictions = eval_prediction.predictions
    labels = eval_prediction.label_ids

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.asarray(predictions)
    labels = np.asarray(labels)

    shifted_predictions = predictions[:, :-1]
    shifted_labels = labels[:, 1:]

    valid_mask = shifted_labels != -100

    if tokenizer.eos_token_id is not None:
        valid_mask &= (
            shifted_labels
            != tokenizer.eos_token_id
        )

    valid_count = int(valid_mask.sum())

    if valid_count:
        correct_count = int(
            (
                shifted_predictions[valid_mask]
                == shifted_labels[valid_mask]
            ).sum()
        )

        label_token_accuracy = (
            correct_count / valid_count
        )
    else:
        label_token_accuracy = 0.0

    metrics = {
        "label_token_accuracy": (
            label_token_accuracy
        )
    }

    losses = getattr(
        eval_prediction,
        "losses",
        None,
    )

    if losses is not None:
        losses = np.asarray(losses)
        mean_loss = float(losses.mean())

        if np.isfinite(mean_loss):
            metrics["perplexity"] = float(
                np.exp(mean_loss)
            )

    return metrics

### 17. Trainer

`Trainer` получает QLoRA-модель, но оптимизатор обновляет только параметры с `requires_grad=True`, то есть LoRA-адаптер.

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_trainer_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

### 18. Обучение

Обучение можно запускать обычным `trainer.train()`. Для продолжения сохранённого запуска используется `trainer.train(resume_from_checkpoint=True)`.

In [18]:
train_result = trainer.train()

trainer.log_metrics(
    "train",
    train_result.metrics,
)

trainer.save_metrics(
    "train",
    train_result.metrics,
)

trainer.save_state()

training_history = trainer.state.log_history

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 248044}.


Step,Training Loss,Validation Loss,Label Token Accuracy,Perplexity
100,0.187188,0.113081,0.912844,1.119723
200,0.091600,0.099874,0.936927,1.105032
300,0.115987,0.098358,0.925459,1.103358
400,0.146311,0.125641,0.913991,1.133875
500,0.082727,0.096654,0.938073,1.101479
600,0.037039,0.126129,0.934633,1.134428
700,0.046257,0.112636,0.946101,1.119225
800,0.039262,0.098535,0.941514,1.103553


***** train metrics *****
  epoch                    =        1.6
  total_flos               =  4848042GF
  train_loss               =      0.124
  train_runtime            = 0:17:41.40
  train_samples_per_second =     22.611
  train_steps_per_second   =      1.413


### 19. Итоговый validation loss

После обучения отдельно выполняется валидация лучшего восстановленного чекпойнта.

In [19]:
eval_metrics = trainer.evaluate()

final_perplexity = eval_metrics.get(
    "eval_perplexity"
)

if final_perplexity is not None:
    final_perplexity = float(final_perplexity)

trainer.log_metrics(
    "eval",
    eval_metrics,
)

trainer.save_metrics(
    "eval",
    eval_metrics,
)

eval_metrics

Training Loss,Validation Loss,Step,Label Token Accuracy,Perplexity
0.039262,0.096654,800,0.938073,1.101479


***** eval metrics *****
  eval_label_token_accuracy = 0.9381
  eval_loss                 = 0.0967
  eval_perplexity           = 1.1015


{'eval_loss': 0.09665380418300629,
 'eval_label_token_accuracy': 0.9380733944954128,
 'eval_perplexity': 1.1014789801909661}

### 20. Динамика обучения

На одном интерактивном Plotly-графике отображаются training loss, validation loss и learning rate. График остаётся только в блокноте и не экспортируется в отдельный файл.

In [20]:
train_loss_points = [
    (entry["epoch"], entry["loss"])
    for entry in training_history
    if (
        "loss" in entry
        and "eval_loss" not in entry
        and "epoch" in entry
    )
]

eval_loss_points = [
    (entry["epoch"], entry["eval_loss"])
    for entry in training_history
    if (
        "eval_loss" in entry
        and "epoch" in entry
    )
]

learning_rate_points = [
    (entry["epoch"], entry["learning_rate"])
    for entry in training_history
    if (
        "learning_rate" in entry
        and "epoch" in entry
    )
]

training_fig = go.Figure()

if train_loss_points:
    train_epochs, train_losses = zip(*train_loss_points)
    training_fig.add_trace(
        go.Scatter(
            x=train_epochs,
            y=train_losses,
            mode="lines+markers",
            name="train loss",
            yaxis="y",
        )
    )

if eval_loss_points:
    eval_epochs, eval_losses = zip(*eval_loss_points)
    training_fig.add_trace(
        go.Scatter(
            x=eval_epochs,
            y=eval_losses,
            mode="lines+markers",
            name="validation loss",
            yaxis="y",
        )
    )

if learning_rate_points:
    lr_epochs, learning_rates = zip(*learning_rate_points)
    training_fig.add_trace(
        go.Scatter(
            x=lr_epochs,
            y=learning_rates,
            mode="lines",
            name="learning rate",
            yaxis="y2",
        )
    )

training_fig.update_layout(
    title="QLoRA training dynamics",
    height=430,
    xaxis={"title": "Epoch"},
    yaxis={"title": "Loss"},
    yaxis2={
        "title": "Learning rate",
        "overlaying": "y",
        "side": "right",
        "tickformat": ".1e",
        "showgrid": False,
    },
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.02,
        "xanchor": "left",
        "x": 0,
    },
    margin={
        "l": 70,
        "r": 90,
        "t": 90,
        "b": 60,
    },
)

training_fig.show()

### 21. Оценка качества после QLoRA

После обучения повторяются оба режима оценки на той же валидационной выборке. Результаты добавляются в `RESULTS_TABLE`, которая затем выводится в разделе `## Результаты`.

In [21]:
model = trainer.model
model.config.use_cache = True

final_generation_metrics = evaluate_generation(
    model,
    dataset["validation"],
    max_samples=FINAL_EVAL_SAMPLES,
    batch_size=GENERATION_BATCH_SIZE,
)

final_forced_choice_metrics = evaluate_forced_choice(
    model,
    dataset["validation"],
    max_samples=FINAL_EVAL_SAMPLES,
    batch_size=FORCED_CHOICE_BATCH_SIZE,
)

assert final_generation_metrics["total"] == len(dataset["validation"])
assert final_forced_choice_metrics["total"] == len(dataset["validation"])

print_classification_summary(
    "QLoRA — generation-based evaluation",
    final_generation_metrics,
)

print_classification_summary(
    "QLoRA — forced-choice evaluation",
    final_forced_choice_metrics,
)

print("\nComparison:")

print(
    "Generation accuracy: "
    f"{baseline_generation_metrics['accuracy']:.2%}"
    " → "
    f"{final_generation_metrics['accuracy']:.2%}"
)

print(
    "Forced-choice accuracy: "
    f"{baseline_forced_choice_metrics['accuracy']:.2%}"
    " → "
    f"{final_forced_choice_metrics['accuracy']:.2%}"
)

print(
    "Generation Macro F1: "
    f"{baseline_generation_metrics['macro_f1']:.4f}"
    " → "
    f"{final_generation_metrics['macro_f1']:.4f}"
)

print(
    "Forced-choice Macro F1: "
    f"{baseline_forced_choice_metrics['macro_f1']:.4f}"
    " → "
    f"{final_forced_choice_metrics['macro_f1']:.4f}"
)

print(
    "Valid output rate: "
    f"{baseline_generation_metrics['valid_output_rate']:.2%}"
    " → "
    f"{final_generation_metrics['valid_output_rate']:.2%}"
)

artifact_expected_generation = (
    final_generation_metrics["predictions"][:ARTIFACT_RELOAD_SAMPLES]
)

artifact_expected_forced_choice = (
    final_forced_choice_metrics["predictions"][:ARTIFACT_RELOAD_SAMPLES]
)

RESULTS_TABLE.append(
    {
        "variant": "QLoRA",
        "generation_accuracy": final_generation_metrics["accuracy"],
        "forced_choice_accuracy": final_forced_choice_metrics["accuracy"],
        "generation_macro_f1": final_generation_metrics["macro_f1"],
        "forced_choice_macro_f1": final_forced_choice_metrics["macro_f1"],
        "valid_output_rate": final_generation_metrics["valid_output_rate"],
    }
)


QLoRA — generation-based evaluation
-----------------------------------
Accuracy:  93.92%
Macro F1:  0.9392
negative precision=0.9310 recall=0.9463 f1=0.9386
positive precision=0.9474 recall=0.9324 f1=0.9398
Valid output rate: 100.00%

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative           405        23         0
positive            30       414         0

QLoRA — forced-choice evaluation
--------------------------------
Accuracy:  93.92%
Macro F1:  0.9392
negative precision=0.9291 recall=0.9486 f1=0.9387
positive precision=0.9494 recall=0.9302 f1=0.9397

Confusion matrix (rows=actual, columns=predicted)
              negative  positive   invalid
negative           406        22         0
positive            31       413         0

Comparison:
Generation accuracy: 0.00% → 93.92%
Forced-choice accuracy: 50.92% → 93.92%
Generation Macro F1: 0.0000 → 0.9392
Forced-choice Macro F1: 0.3374 → 0.9392
Valid output rate: 0.00% → 100.00%


### 22. Сохранение QLoRA-адаптера

Сохраняется только PEFT-адаптер и токенизатор. 4-битная базовая модель не копируется в каталог результата и при загрузке берётся из исходного репозитория.

In [22]:
model.save_pretrained(
    OUTPUT_DIR,
    safe_serialization=True,
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)

print(f"Saved QLoRA adapter to: {OUTPUT_DIR}")

Saved QLoRA adapter to: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-qlora


### 23. Размер адаптера

Размер считается только по файлам адаптера и не включает чекпойнты `Trainer` или исходную модель.

In [23]:
adapter_files = [
    OUTPUT_DIR / "adapter_model.safetensors",
    OUTPUT_DIR / "adapter_config.json",
]

adapter_size_bytes = sum(
    path.stat().st_size
    for path in adapter_files
    if path.exists()
)

print(
    "QLoRA adapter size:",
    f"{adapter_size_bytes / 1024**2:.3f} MiB",
)

print("\nFinal export files:")

for file in sorted(OUTPUT_DIR.iterdir()):
    if file.is_file():
        print(
            f"{file.name:32s} "
            f"{file.stat().st_size / 1024:.1f} KiB"
        )

QLoRA adapter size: 64.209 MiB

Final export files:
README.md                        5.1 KiB
adapter_config.json              1.2 KiB
adapter_model.safetensors        65749.1 KiB
chat_template.jinja              7.6 KiB
tokenizer.json                   19521.1 KiB
tokenizer_config.json            1.1 KiB


### 24. Метаданные воспроизводимости

Отдельный файл фиксирует конфигурацию конкретного запуска: ревизии модели и набора данных, параметры NF4, LoRA и основные параметры обучения.

In [24]:
REPRODUCIBILITY_PATH = (
    OUTPUT_DIR / "reproducibility.json"
)

reproducibility_data = {
    "seed": SEED,
    "run_mode": RUN_MODE,
    "model_id": MODEL_ID,
    "model_revision_requested": MODEL_REVISION,
    "model_revision_resolved": resolved_model_revision,
    "dataset_id": DATASET_ID,
    "dataset_revision_requested": DATASET_REVISION,
    "dataset_revision_resolved": resolved_dataset_revision,
    "train_fingerprint": dataset_train_fingerprint,
    "validation_fingerprint": dataset_validation_fingerprint,
    "train_examples": len(dataset["train"]),
    "validation_examples": len(dataset["validation"]),
    "quantization": {
        "load_in_4bit": True,
        "quant_type": BNB_QUANT_TYPE,
        "compute_dtype": str(BNB_COMPUTE_DTYPE),
        "double_quant": BNB_USE_DOUBLE_QUANT,
    },
    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGET_MODULES,
    },
    "training": {
        "epochs": NUM_TRAIN_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "lr_scheduler_type": LR_SCHEDULER_TYPE,
        "warmup_ratio": WARMUP_RATIO,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "optimizer": "paged_adamw_32bit",
        "eval_steps": EVAL_STEPS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    },
}

REPRODUCIBILITY_PATH.write_text(
    json.dumps(
        reproducibility_data,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    "Saved reproducibility metadata:",
    REPRODUCIBILITY_PATH,
)

Saved reproducibility metadata: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-qlora/reproducibility.json


### 25. Проверка сохранённого адаптера

Для проверки создаётся новая 4-битная NF4-модель, поверх неё загружается сохранённый адаптер, а предсказания на фиксированной подвыборке сравниваются с результатом до сохранения.

In [25]:
def clear_device_cache():
    gc.collect()
    torch.cuda.empty_cache()


if RUN_ARTIFACT_RELOAD_TEST:
    del model
    del base_model
    clear_device_cache()

    adapter_config = PeftConfig.from_pretrained(
        OUTPUT_DIR
    )

    reload_quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=BNB_QUANT_TYPE,
        bnb_4bit_compute_dtype=BNB_COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=BNB_USE_DOUBLE_QUANT,
    )

    reloaded_base_model = AutoModelForCausalLM.from_pretrained(
        adapter_config.base_model_name_or_path,
        revision=MODEL_REVISION,
        dtype=torch.bfloat16,
        quantization_config=reload_quantization_config,
        device_map={"": torch.cuda.current_device()},
    )

    reloaded_model = PeftModel.from_pretrained(
        reloaded_base_model,
        OUTPUT_DIR,
    )

    reloaded_model.eval()

    reload_generation_metrics = evaluate_generation(
        reloaded_model,
        dataset["validation"],
        max_samples=ARTIFACT_RELOAD_SAMPLES,
        batch_size=ARTIFACT_RELOAD_SAMPLES,
    )

    reload_forced_choice_metrics = evaluate_forced_choice(
        reloaded_model,
        dataset["validation"],
        max_samples=ARTIFACT_RELOAD_SAMPLES,
        batch_size=ARTIFACT_RELOAD_SAMPLES,
    )

    assert (
        reload_generation_metrics["predictions"]
        == artifact_expected_generation
    )

    assert (
        reload_forced_choice_metrics["predictions"]
        == artifact_expected_forced_choice
    )

    print("Adapter reload smoke test: PASSED")
    print(f"Checked examples: {ARTIFACT_RELOAD_SAMPLES}")

    del reload_generation_metrics
    del reload_forced_choice_metrics
    del reloaded_model
    del reloaded_base_model
    del adapter_config
    clear_device_cache()
else:
    print("RUN_ARTIFACT_RELOAD_TEST=False — reload test skipped.")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Adapter reload smoke test: PASSED
Checked examples: 8


### 26. Создание Hugging Face Model Card

Model Card создаётся для адаптера и описывает базовую модель, 4-битную схему NF4, конфигурацию LoRA и фактические результаты оценки.

In [26]:
MODEL_CARD_PATH = OUTPUT_DIR / "README.md"

model_display_name = HUB_MODEL_ID.split("/")[-1]

perplexity_text = (
    f"{final_perplexity:.4f}"
    if final_perplexity is not None
    else "n/a"
)

card_text = f"""---
base_model: {MODEL_ID}
library_name: peft
pipeline_tag: text-generation
datasets:
- {DATASET_ID}
language:
- en
license: apache-2.0
tags:
- peft
- qlora
- lora
- bitsandbytes
- nf4
- qwen3.5
- sentiment-analysis
---

# {model_display_name}

QLoRA adapter for `{MODEL_ID}`, fine-tuned on `{DATASET_ID}`
for binary sentiment classification.

## Model Details

| Property | Value |
|---|---|
| Base model | `{MODEL_ID}` |
| Method | QLoRA |
| Base quantization | NF4, 4-bit |
| Double quantization | {BNB_USE_DOUBLE_QUANT} |
| Compute dtype | BF16 |
| LoRA rank | {LORA_R} |
| LoRA alpha | {LORA_ALPHA} |
| LoRA dropout | {LORA_DROPOUT} |
| Target modules | `{LORA_TARGET_MODULES}` |
| Dataset | `{DATASET_ID}` |
| Adapter size | {adapter_size_bytes / 1024**2:.2f} MiB |

## Evaluation

| Metric | NF4 base | QLoRA |
|---|---:|---:|
| Generation accuracy | {baseline_generation_metrics["accuracy"]:.2%} | **{final_generation_metrics["accuracy"]:.2%}** |
| Forced-choice accuracy | {baseline_forced_choice_metrics["accuracy"]:.2%} | **{final_forced_choice_metrics["accuracy"]:.2%}** |
| Generation Macro F1 | {baseline_generation_metrics["macro_f1"]:.4f} | **{final_generation_metrics["macro_f1"]:.4f}** |
| Forced-choice Macro F1 | {baseline_forced_choice_metrics["macro_f1"]:.4f} | **{final_forced_choice_metrics["macro_f1"]:.4f}** |
| Valid output rate | {baseline_generation_metrics["valid_output_rate"]:.2%} | **{final_generation_metrics["valid_output_rate"]:.2%}** |
| Perplexity | — | {perplexity_text} |

## Usage

```python
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL_ID = "{MODEL_ID}"
ADAPTER_ID = "{HUB_MODEL_ID}"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.bfloat16,
    quantization_config=quantization_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_ID,
)

model.eval()
```

## Limitations

- The adapter is trained for English SST-2 sentiment classification.
- Inference requires the original base model and the same 4-bit loading scheme.
"""

MODEL_CARD_PATH.write_text(
    card_text,
    encoding="utf-8",
)

print(f"Saved Model Card: {MODEL_CARD_PATH}")

Saved Model Card: /workspace/notebooks/finetuning/peft/outputs/qwen3.5-2b-sst2-qlora/README.md


### 27. Публикация в Hugging Face Hub

Публикация отключена по умолчанию. При `PUSH_TO_HUB=True` отправляются адаптер, токенизатор, Model Card и метаданные воспроизводимости; чекпойнты обучения исключаются.

In [27]:
if PUSH_TO_HUB:
    create_repo(
        repo_id=HUB_MODEL_ID,
        repo_type="model",
        exist_ok=True,
    )

    api = HfApi()

    commit_info = api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HUB_MODEL_ID,
        repo_type="model",
        ignore_patterns=[
            "checkpoints/**",
            "checkpoint-*/**",
            "runs/**",
            "*.pt",
            "*.pth",
        ],
        commit_message=(
            "Upload Qwen3.5-2B SST-2 QLoRA adapter "
            "and model card"
        ),
    )

    print(
        f"Published: "
        f"https://huggingface.co/{HUB_MODEL_ID}"
    )

    print(f"Commit: {commit_info.commit_url}")
else:
    print("PUSH_TO_HUB=False - nothing was uploaded.")

PUSH_TO_HUB=False - nothing was uploaded.


## Результаты

### Итоговая сравнительная таблица

In [28]:
assert [row["variant"] for row in RESULTS_TABLE] == [
    "Исходная NF4",
    "QLoRA",
]

print(
    f"{'Вариант':18s}"
    f"{'Generation Accuracy':>22s}"
    f"{'Forced-Choice Accuracy':>24s}"
    f"{'Generation Macro F1':>22s}"
    f"{'Forced-Choice Macro F1':>24s}"
    f"{'Valid Output Rate':>20s}"
)

print("-" * 130)

for row in RESULTS_TABLE:
    print(
        f"{row['variant']:18s}"
        f"{row['generation_accuracy']:22.2%}"
        f"{row['forced_choice_accuracy']:24.2%}"
        f"{row['generation_macro_f1']:22.4f}"
        f"{row['forced_choice_macro_f1']:24.4f}"
        f"{row['valid_output_rate']:20.2%}"
    )

Вариант              Generation Accuracy  Forced-Choice Accuracy   Generation Macro F1  Forced-Choice Macro F1   Valid Output Rate
----------------------------------------------------------------------------------------------------------------------------------
Исходная NF4                       0.00%                  50.92%                0.0000                  0.3374               0.00%
QLoRA                             93.92%                  93.92%                0.9392                  0.9392             100.00%


Сравнительная таблица показывает существенное улучшение качества после QLoRA сразу в обоих режимах оценки.

У исходной 4-битной NF4-модели **Generation Accuracy составляет 0.00%**, а **Valid Output Rate — 0.00%**. При свободной генерации модель ни разу не выдала ответ в требуемом формате `positive` или `negative`, поэтому Generation Accuracy в данном случае не отражает напрямую способность модели распознавать тональность.

**Forced-Choice Accuracy исходной NF4-модели составляет 50.92%**, а **Forced-Choice Macro F1 — 0.3374**. Accuracy находится практически на уровне случайного выбора для бинарной классификации, а низкий Macro F1 указывает на выраженный перекос в сторону одного из классов. Следовательно, исходная квантизованная модель без адаптации не только не соблюдает формат ответа, но и слабо разделяет классы SST-2 в выбранной постановке задачи.

После QLoRA **Generation Accuracy возрастает до 93.92%**, а **Forced-Choice Accuracy — также до 93.92%**. Прирост относительно исходной NF4-модели составляет соответственно **+93.92** и **+43.00 процентного пункта**. При этом **Generation Macro F1 увеличивается с 0.0000 до 0.9392**, а **Forced-Choice Macro F1 — с 0.3374 до 0.9392**.

**Valid Output Rate возрастает с 0.00% до 100.00%**, то есть после QLoRA модель стабильно соблюдает требуемый формат ответа. Особенно показательно, что после адаптации **Generation Accuracy и Forced-Choice Accuracy полностью совпадают — 93.92%**, как и соответствующие значения **Macro F1 — 0.9392**. Это означает, что свободная генерация больше не вносит дополнительной ошибки относительно принудительного выбора класса.

Таким образом, в этом эксперименте QLoRA успешно решила две задачи одновременно: существенно улучшила качество бинарной классификации SST-2 и сформировала устойчивое поведение при генерации ответа. При этом базовая модель остаётся загруженной в 4-битном формате NF4 и не обновляется, а обучение выполняется только для компактного LoRA-адаптера. Полученный результат демонстрирует, что 4-битное представление базовой модели не препятствует эффективной адаптации к целевой задаче.

## Источники

- Dettmers et al. — QLoRA: Efficient Finetuning of Quantized LLMs: https://arxiv.org/abs/2305.14314
- Hugging Face PEFT 0.20.0 — Quantization: https://huggingface.co/docs/peft/v0.20.0/developer_guides/quantization
- Hugging Face PEFT — LoRA: https://huggingface.co/docs/peft/main/package_reference/lora
- Hugging Face Transformers — bitsandbytes: https://huggingface.co/docs/transformers/main/quantization/bitsandbytes
- bitsandbytes — 4-bit quantization: https://huggingface.co/docs/bitsandbytes/main/reference/nn/linear4bit
- Qwen3.5-2B-Base: https://huggingface.co/Qwen/Qwen3.5-2B-Base
- Stanford SST-2: https://huggingface.co/datasets/stanfordnlp/sst2